# MiVOLO V2 → Jetson용 TorchScript 변환

이 노트북은 Hugging Face의 동적 모듈 경로에 포함된 숫자 시작 커밋 해시를 TorchScript가 읽을 수 있는 이름으로 바꾼 뒤 모델을 저장합니다.

생성 파일: `mivolo_v2_torchscript_jetson.pt`


In [1]:
# 필요한 패키지 설치
!pip install -q "transformers==4.51.0" "accelerate==1.8.1" "timm==0.8.13.dev0"
!pip install -q --no-deps git+https://github.com/WildChlamydia/MiVOLO.git


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Preparing metadata (setup.py) ... done


In [2]:
import re
import sys
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoModelForImageClassification

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [8]:
MODEL_ID = "iitolstykh/mivolo_v2"
OUTPUT_PATH = Path("mivolo_v2.pt")


class MiVOLOAgeWrapper(nn.Module):
    """MiVOLO 출력 중 age_output만 반환합니다."""

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(
        self,
        face_input: torch.Tensor,
        body_input: torch.Tensor,
    ) -> torch.Tensor:
        outputs = self.model(
            faces_input=face_input,
            body_input=body_input,
            return_dict=False,
        )
        return outputs[1]


def sanitize_torchscript_module_names(model: nn.Module) -> None:
    """
    예:
      transformers_modules.xxx.mivolo_v2.53393526....modeling_mivolo
    변경:
      transformers_modules.xxx.mivolo_v2._53393526....modeling_mivolo

    Python/TorchScript 식별자는 숫자로 시작할 수 없으므로 '_'를 붙입니다.
    """
    changed_classes = set()

    for submodule in model.modules():
        cls = submodule.__class__
        if cls in changed_classes:
            continue

        old_name = cls.__module__
        parts = old_name.split(".")
        safe_parts = [
            f"_{part}" if part and part[0].isdigit() else part
            for part in parts
        ]
        new_name = ".".join(safe_parts)

        if new_name != old_name:
            cls.__module__ = new_name
            changed_classes.add(cls)
            print("Sanitized module name:")
            print("  before:", old_name)
            print("  after :", new_name)


In [9]:
# MiVOLO 모델 로드
base_model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float32,
)

base_model.eval()
base_model.cpu()
base_model.float()

# TorchScript 저장 전에 숫자 시작 네임스페이스를 수정합니다.
sanitize_torchscript_module_names(base_model)

wrapper = MiVOLOAgeWrapper(base_model).eval()
print("MiVOLO V2 loaded")


MiVOLO V2 loaded


In [10]:
# TorchScript 변환 및 저장
dummy_face = torch.zeros((1, 3, 384, 384), dtype=torch.float32)
dummy_body = torch.zeros((1, 3, 384, 384), dtype=torch.float32)

with torch.inference_mode():
    traced_model = torch.jit.trace(
        wrapper,
        (dummy_face, dummy_body),
        strict=False,
        check_trace=False,
    )
    test_age = traced_model(dummy_face, dummy_body)

traced_model.save(str(OUTPUT_PATH))

print("Saved:", OUTPUT_PATH.resolve())
print("Output shape:", tuple(test_age.shape))
print("File size:", f"{OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")


Saved: /content/mivolo_v2.pt
Output shape: (1, 1)
File size: 110.48 MB


In [11]:
# 저장된 TorchScript를 다시 열어 검증합니다.
reloaded = torch.jit.load(str(OUTPUT_PATH), map_location="cpu")
reloaded.eval()

with torch.inference_mode():
    reload_age = reloaded(dummy_face, dummy_body)

print("Reload test passed")
print("Reload output shape:", tuple(reload_age.shape))

# 압축 내부 Python 코드에서 '.숫자' 형태의 잘못된 네임스페이스가
# 남아 있는지 검사합니다.
invalid_pattern = re.compile(r"\.[0-9][A-Za-z0-9_]*\.")
invalid_entries = []

with zipfile.ZipFile(OUTPUT_PATH, "r") as archive:
    for name in archive.namelist():
        if not name.endswith(".py"):
            continue
        source = archive.read(name).decode("utf-8", errors="ignore")
        if invalid_pattern.search(source):
            invalid_entries.append(name)

if invalid_entries:
    raise RuntimeError(
        "Invalid numeric TorchScript namespace remains: "
        + ", ".join(invalid_entries)
    )

print("Namespace validation passed")


Reload test passed
Reload output shape: (1, 1)
Namespace validation passed


In [7]:
# 검증된 파일 다운로드
from google.colab import files

files.download(str(OUTPUT_PATH))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>